# SAGE-Credit Experiment

Trains the CatBoost experts and neural gate, evaluates score-availability conditions, and produces the final analyses.

This is the publication copy for the SAGE-Credit repository. The original analysis logic is preserved, machine-specific paths are replaced with portable configuration, and cached outputs are cleared.

> Download the Home Credit Default Risk competition files separately; the dataset is not redistributed here.

In [ ]:
from pathlib import Path
import os

# Portable repository configuration. Override either location with an environment variable.
WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
DATA_DIR = Path(os.environ.get("HOME_CREDIT_DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser().resolve()
ARTIFACT_DIR = Path(os.environ.get("SAGE_CREDIT_ARTIFACT_DIR", PROJECT_ROOT / "artifacts")).expanduser().resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {DATA_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
# Import the libraries used for data handling, modeling, evaluation, and saving artifacts.
from pathlib import Path
import gc
import json
import random
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore', category=FutureWarning)


def clean():
    """Release objects that are no longer referenced by the notebook."""
    gc.collect()


# One seed controls every deterministic split, scenario ranking, and model fit.
SEED = 42
N_FOLDS = 3
BASE_ITERATIONS = 2200

APPLICATION_ONLY_ITERATIONS = BASE_ITERATIONS
GATE_MAX_EPOCHS = 60
GATE_PATIENCE = 10
BOOTSTRAP_ITERATIONS = 5000

# This seed controls which available external scores are retained per applicant.
SCENARIO_RANKING_SEED = 4200

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
# Input files are read from their existing locations. Every artifact created by
# this notebook is written to the original research-project output directory.
TABULAR_PATH = Path(ARTIFACT_DIR / "home_credit_tabular_features.pkl")
APPLICATION_PATH = Path(DATA_DIR / "application_train.csv")
OUTPUT_DIR = ARTIFACT_DIR / "sage_credit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for label, file_path in {
    'Tabular pickle': TABULAR_PATH,
    'Raw application file': APPLICATION_PATH,
}.items():
    if not file_path.exists():
        raise FileNotFoundError(f'{label} not found: {file_path}')


In [ ]:
master = pd.read_pickle(TABULAR_PATH)

required_master_columns = {'SK_ID_CURR', 'TARGET'}
missing_required = required_master_columns - set(master.columns)
if missing_required:
    raise ValueError(f'Tabular pickle is missing: {sorted(missing_required)}')

if master['SK_ID_CURR'].duplicated().any():
    raise ValueError('The tabular dataset contains duplicate applicant IDs.')

applicants = master['SK_ID_CURR'].to_numpy(copy=True)
y = master['TARGET'].to_numpy(dtype=np.int8, copy=True)
X = master.drop(columns=['SK_ID_CURR', 'TARGET']).copy()
del master
clean()

def sanitize_unique_columns(columns):
    cleaned = [re.sub(r'[^A-Za-z0-9_]+', '_', str(col)) for col in columns]
    counts = {}
    unique = []
    for col in cleaned:
        number = counts.get(col, 0)
        unique.append(col if number == 0 else f'{col}__{number}')
        counts[col] = number + 1
    return unique

X.columns = sanitize_unique_columns(X.columns)
X = X.replace([np.inf, -np.inf], np.nan)

assert len(X) == 307_511
assert X.shape[1] == 929
assert len(np.unique(applicants)) == len(applicants)


In [ ]:
RAW_SCORE_COLUMNS = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
DERIVED_SCORE_COLUMNS = [
    'APP_EXT_SOURCE_MEAN',
    'APP_EXT_SOURCE_MIN',
    'APP_EXT_SOURCE_MAX',
    'APP_EXT_SOURCE_STD',
]
ALL_SCORE_COLUMNS = RAW_SCORE_COLUMNS + DERIVED_SCORE_COLUMNS

raw_scores = pd.read_csv(
    APPLICATION_PATH,
    usecols=['SK_ID_CURR'] + RAW_SCORE_COLUMNS,
).set_index('SK_ID_CURR')

if not raw_scores.index.is_unique:
    raise ValueError('Raw application file contains duplicate applicant IDs.')
missing_applicants = pd.Index(applicants).difference(raw_scores.index)
if len(missing_applicants) > 0:
    raise ValueError(f'Raw application file is missing {len(missing_applicants)} tabular applicants.')

raw_scores = raw_scores.reindex(applicants)
if len(raw_scores) != len(applicants):
    raise ValueError('Could not align raw application scores to the tabular applicants.')

for column in RAW_SCORE_COLUMNS:
    X[column] = raw_scores[column].to_numpy(dtype=np.float32)

def recompute_score_summaries(frame):
    score_block = frame[RAW_SCORE_COLUMNS]
    frame['APP_EXT_SOURCE_MEAN'] = score_block.mean(axis=1, skipna=True)
    frame['APP_EXT_SOURCE_MIN'] = score_block.min(axis=1, skipna=True)
    frame['APP_EXT_SOURCE_MAX'] = score_block.max(axis=1, skipna=True)
    frame['APP_EXT_SOURCE_STD'] = score_block.std(axis=1, skipna=True)
    return frame

X = recompute_score_summaries(X)

del raw_scores
clean()


In [ ]:
HISTORY_PREFIXES = {
    'CC': 'CC_',
    'POS': 'POS_',
    'INST': 'INST_',
    'PREV': 'PREV_',
    'BUR': 'BUR_',
    'BB': 'BB_',
}

# Preserve the original 19 gate inputs: five sources × two context summaries.
GATE_HISTORY_PREFIXES = {
    source: prefix for source, prefix in HISTORY_PREFIXES.items() if source != 'BB'
}

missing_score_features = [col for col in ALL_SCORE_COLUMNS if col not in X.columns]
if missing_score_features:
    raise ValueError(f'Expected external-score features not found: {missing_score_features}')

full_feature_columns = list(X.columns)
score_free_columns = [col for col in X.columns if col not in ALL_SCORE_COLUMNS]
history_columns = [
    col for col in score_free_columns
    if any(col.startswith(prefix) for prefix in HISTORY_PREFIXES.values())
]
application_only_columns = [col for col in score_free_columns if col not in history_columns]

assert len(full_feature_columns) == 929
assert len(score_free_columns) == 922
assert len(history_columns) == 675
assert len(application_only_columns) == 247
assert not any(col.startswith('BB_') for col in application_only_columns)
assert len(application_only_columns) + len(history_columns) + len(ALL_SCORE_COLUMNS) == 929


In [ ]:
# First reserve the final 10% test set. Features, targets, and applicant IDs are
# split together so their row order cannot become misaligned.
original_applicant_count = len(applicants)

X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    applicants,
    test_size=0.10,
    random_state=SEED,
    stratify=y,
)

# Split the remaining 90% into permanent train and validation partitions.
# The 0.1111 fraction preserves the original experiment's 30,748-row validation set.
X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=0.1111,
    random_state=SEED,
    stratify=y_train_val,
)

# Positional indices are required later when OOF fold positions select DataFrame rows.
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

# Verify row alignment and the intended approximately 80/10/10 population split.
assert len(X_train) == len(y_train) == len(ids_train) == 246_011
assert len(X_val) == len(y_val) == len(ids_val) == 30_748
assert len(X_test) == len(y_test) == len(ids_test) == 30_752

# No applicant may appear in more than one partition.
assert set(ids_train).isdisjoint(ids_val)
assert set(ids_train).isdisjoint(ids_test)
assert set(ids_val).isdisjoint(ids_test)

# Every original applicant must appear exactly once across the three partitions.
all_split_ids = np.concatenate([ids_train, ids_val, ids_test])
assert len(all_split_ids) == original_applicant_count
assert len(np.unique(all_split_ids)) == original_applicant_count

# Save the exact applicant partition so every later result can be audited.
np.savez_compressed(
    OUTPUT_DIR / 'split_ids.npz',
    ids_train=ids_train,
    ids_val=ids_val,
    ids_test=ids_test,
)

# Keep all three final partitions; remove only the unsplit and temporary containers.
del X, y, applicants, X_train_val, y_train_val, ids_train_val, all_split_ids
clean()


In [ ]:
SCENARIOS = {
    'natural': None,
    'max_2_scores': 2,
    'max_1_score': 1,
    'no_scores': 0,
}


def stable_score_keys(applicant_ids, seed):
    """
    Create deterministic pseudo-random rankings for the three external scores.

    np.uint64 stores unsigned 64-bit integers. The arithmetic intentionally wraps
    at 2**64, which is a standard hashing technique. It gives each applicant a
    repeatable score preference without consulting a target or model prediction.
    """
    ids = np.asarray(applicant_ids, dtype=np.uint64)
    keys = np.empty((len(ids), len(RAW_SCORE_COLUMNS)), dtype=np.uint64)
    mask = np.uint64(0xFFFFFFFFFFFFFFFF)

    # Overflow is expected because the hash deliberately operates modulo 2**64.
    with np.errstate(over='ignore'):
        for column_number in range(len(RAW_SCORE_COLUMNS)):
            column_offset = np.multiply(
                np.uint64(column_number + 1),
                np.uint64(0x9E3779B97F4A7C15),
                dtype=np.uint64,
            )
            value = (ids + np.uint64(seed) + column_offset) & mask
            value = (
                (value ^ (value >> np.uint64(30)))
                * np.uint64(0xBF58476D1CE4E5B9)
            ) & mask
            value = (
                (value ^ (value >> np.uint64(27)))
                * np.uint64(0x94D049BB133111EB)
            ) & mask
            keys[:, column_number] = value ^ (value >> np.uint64(31))

    return keys


def apply_score_scenario(frame, maximum_scores, applicant_ids, seed=SCENARIO_RANKING_SEED):
    """
    Retain at most the requested number of naturally available external scores.

    The four derived score summaries are recalculated from the scores that remain.
    Consequently, one or two retained scores still produce their available summaries,
    while removing all three raw scores makes all four derived summaries NaN.
    """
    if len(frame) != len(applicant_ids):
        raise ValueError('Frame and applicant IDs must have identical row counts.')

    changed = frame.copy()
    available = changed[RAW_SCORE_COLUMNS].notna().to_numpy()

    if maximum_scores is None:
        keep = available.copy()
    elif maximum_scores == 0:
        keep = np.zeros_like(available, dtype=bool)
    else:
        keys = stable_score_keys(applicant_ids, seed)

        # An originally missing score can never be selected for retention.
        keys[~available] = np.uint64(0)
        preference_order = np.argsort(keys, axis=1)[:, ::-1]
        keep = np.zeros_like(available, dtype=bool)
        rows = np.arange(len(changed))

        for rank in range(min(maximum_scores, len(RAW_SCORE_COLUMNS))):
            chosen_columns = preference_order[:, rank]
            valid_rows = available[rows, chosen_columns]
            keep[rows[valid_rows], chosen_columns[valid_rows]] = True

    # Only originally available scores that were not selected are removed.
    for column_number, column in enumerate(RAW_SCORE_COLUMNS):
        remove = available[:, column_number] & ~keep[:, column_number]
        changed.loc[remove, column] = np.nan

    # Recalculate the four summaries from the retained raw scores.
    changed = recompute_score_summaries(changed)
    return changed, keep


# Validate the deterministic nested scenarios using training data only.
# The test features are deliberately not inspected here.
scenario_availability_check = {}
for scenario_name, cap in SCENARIOS.items():
    transformed, availability = apply_score_scenario(
        X_train[ALL_SCORE_COLUMNS],
        maximum_scores=cap,
        applicant_ids=ids_train,
    )
    assert np.array_equal(
        availability,
        transformed[RAW_SCORE_COLUMNS].notna().to_numpy(),
    ), f'Mask mismatch in {scenario_name}'
    if cap is not None:
        assert (availability.sum(axis=1) <= cap).all()
    if scenario_name == 'no_scores':
        assert transformed[ALL_SCORE_COLUMNS].isna().all().all()
    scenario_availability_check[scenario_name] = availability

# Each stricter scenario must be a subset of the preceding scenario.
assert np.all(
    scenario_availability_check['max_1_score']
    <= scenario_availability_check['max_2_scores']
)
assert np.all(
    scenario_availability_check['max_2_scores']
    <= scenario_availability_check['natural']
)
assert not scenario_availability_check['no_scores'].any()

del scenario_availability_check, transformed, availability
clean()


In [ ]:
def clean_probabilities(probabilities):
    return np.clip(np.asarray(probabilities, dtype=np.float64), 1e-6, 1 - 1e-6)


def probability_logit(probabilities):
    p = clean_probabilities(probabilities)
    return np.log(p / (1 - p))


def paired_auc_bootstrap(labels, first, second, iterations=5000, seed=42):
    labels = np.asarray(labels)
    first = clean_probabilities(first)
    second = clean_probabilities(second)
    rng = np.random.default_rng(seed)

    observed_delta = roc_auc_score(labels, first) - roc_auc_score(labels, second)
    differences = []
    for _ in range(iterations):
        sample = rng.integers(0, len(labels), size=len(labels))
        sampled_labels = labels[sample]
        if np.unique(sampled_labels).size < 2:
            continue
        differences.append(
            roc_auc_score(sampled_labels, first[sample])
            - roc_auc_score(sampled_labels, second[sample])
        )

    return {
        'Observed AUC delta': float(observed_delta),
        'AUC CI lower': float(np.quantile(differences, 0.025)),
        'AUC CI upper': float(np.quantile(differences, 0.975)),
        'Bootstrap iterations completed': len(differences),
    }


def make_catboost(iterations, seed):
    return CatBoostClassifier(
        iterations=iterations,
        learning_rate=0.03,
        depth=4,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=seed,
        colsample_bylevel=0.8,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        verbose=500,
        allow_writing_files=False,
        thread_count=-1,
    )


In [ ]:
def build_history_context(frame):
    """Summarize whether each historical source is present for each applicant."""
    context = {}
    for source, prefix in GATE_HISTORY_PREFIXES.items():
        columns = [col for col in history_columns if col.startswith(prefix)]
        if not columns:
            context[f'{source}_present'] = np.zeros(len(frame), dtype=np.float32)
            context[f'{source}_nonzero_fraction'] = np.zeros(len(frame), dtype=np.float32)
            continue

        source_values = frame[columns].fillna(0).to_numpy(dtype=np.float32, copy=False)
        nonzero_fraction = np.count_nonzero(source_values, axis=1) / len(columns)
        context[f'{source}_present'] = (nonzero_fraction > 0).astype(np.float32)
        context[f'{source}_nonzero_fraction'] = nonzero_fraction.astype(np.float32)

    return pd.DataFrame(context)


def build_gate_features(full_predictions, score_free_predictions, score_availability, history_context):
    """Combine expert behavior, score availability, and history coverage for the gate."""
    full_predictions = clean_probabilities(full_predictions)
    score_free_predictions = clean_probabilities(score_free_predictions)
    availability = pd.DataFrame(
        np.asarray(score_availability, dtype=np.float32),
        columns=RAW_SCORE_COLUMNS,
    )

    gate_features = pd.DataFrame({
        'full_logit': probability_logit(full_predictions),
        'score_free_logit': probability_logit(score_free_predictions),
        'expert_probability_gap': np.abs(full_predictions - score_free_predictions),
        'full_confidence': np.abs(full_predictions - 0.5),
        'score_free_confidence': np.abs(score_free_predictions - 0.5),
        'scores_available': availability.sum(axis=1).to_numpy(dtype=np.float32),
    })

    for column in RAW_SCORE_COLUMNS:
        gate_features[f'{column}_available'] = availability[column].to_numpy(dtype=np.float32)

    gate_features = pd.concat(
        [gate_features.reset_index(drop=True), history_context.reset_index(drop=True)],
        axis=1,
    )
    return gate_features.astype(np.float32)


# These contexts are deterministic feature summaries, not fitted transformations.
history_context_train = build_history_context(X_train)
history_context_val = build_history_context(X_val)


## Out-of-fold training


In [ ]:
OOF_PATH = OUTPUT_DIR / 'aligned_oof_predictions.npz'

def generate_oof_predictions():
    oof_full = np.full((len(y_train), len(SCENARIOS)), np.nan, dtype=np.float32)
    oof_score_free = np.full(len(y_train), np.nan, dtype=np.float32)
    oof_fold = np.full(len(y_train), -1, dtype=np.int8)
    oof_score_availability = np.zeros(
        (len(y_train), len(SCENARIOS), len(RAW_SCORE_COLUMNS)),
        dtype=bool,
    )

    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (fit_positions, holdout_positions) in enumerate(splitter.split(X_train, y_train), start=1):
        print(f'OOF fold {fold}/{N_FOLDS}')
        full_fold_model = make_catboost(BASE_ITERATIONS, SEED + fold)
        full_fold_model.fit(X_train.iloc[fit_positions][full_feature_columns], y_train[fit_positions])
        score_free_fold_model = make_catboost(BASE_ITERATIONS, SEED + 100 + fold)
        score_free_fold_model.fit(X_train.iloc[fit_positions][score_free_columns], y_train[fit_positions])

        assert np.intersect1d(fit_positions, holdout_positions).size == 0
        assert np.all(oof_fold[holdout_positions] == -1)
        oof_fold[holdout_positions] = fold

        holdout_original = X_train.iloc[holdout_positions].reset_index(drop=True)
        holdout_ids = ids_train[holdout_positions]
        oof_score_free[holdout_positions] = score_free_fold_model.predict_proba(
            holdout_original[score_free_columns]
        )[:, 1]

        for scenario_number, (scenario_name, cap) in enumerate(SCENARIOS.items()):
            transformed, availability = apply_score_scenario(
                holdout_original,
                maximum_scores=cap,
                applicant_ids=holdout_ids,
            )
            assert np.array_equal(
                availability,
                transformed[RAW_SCORE_COLUMNS].notna().to_numpy(),
            ), f'OOF mask mismatch in fold {fold}, scenario {scenario_name}'
            oof_full[holdout_positions, scenario_number] = full_fold_model.predict_proba(
                transformed[full_feature_columns]
            )[:, 1]
            oof_score_availability[holdout_positions, scenario_number] = availability
            del transformed

        del full_fold_model, score_free_fold_model, holdout_original
        clean()

    np.savez_compressed(
        OOF_PATH,
        oof_full=oof_full,
        oof_score_free=oof_score_free,
        oof_score_availability=oof_score_availability,
        oof_fold=oof_fold,
        y_train=y_train,
        ids_train=ids_train,
        scenario_names=np.array(list(SCENARIOS)),
        alignment_version=np.array('leakage_safe_train_validation_test'),
    )
    return oof_full, oof_score_free, oof_score_availability, oof_fold

oof_full, oof_score_free, oof_score_availability, oof_fold = generate_oof_predictions()

assert np.all(oof_fold >= 1) and np.all(oof_fold <= N_FOLDS)
assert np.all(oof_score_availability[:, 2] <= oof_score_availability[:, 1])
assert np.all(oof_score_availability[:, 1] <= oof_score_availability[:, 0])
assert not oof_score_availability[:, 3].any()
assert oof_full.shape == (len(y_train), len(SCENARIOS))
assert oof_score_availability.shape == (len(y_train), len(SCENARIOS), 3)
assert np.isfinite(oof_full).all() and np.isfinite(oof_score_free).all()
assert ((oof_full >= 0) & (oof_full <= 1)).all()
assert ((oof_score_free >= 0) & (oof_score_free <= 1)).all()
print('OOF predictions and exact nested availability masks:', oof_full.shape, oof_score_availability.shape)


## OOF-tuned fixed blend


In [ ]:
def select_global_fixed_blend_weight(
    labels,
    oof_full_predictions,
    oof_score_free_predictions,
):
    weight_grid = np.linspace(0.0, 1.0, 101)
    mean_scenario_auc = np.empty(len(weight_grid), dtype=np.float64)

    for weight_number, weight in enumerate(weight_grid):
        scenario_aucs = []
        for scenario_number in range(oof_full_predictions.shape[1]):
            blend = (
                weight * oof_full_predictions[:, scenario_number]
                + (1.0 - weight) * oof_score_free_predictions
            )
            scenario_aucs.append(roc_auc_score(labels, blend))
        mean_scenario_auc[weight_number] = np.mean(scenario_aucs)

    best_index = int(np.argmax(mean_scenario_auc))
    return float(weight_grid[best_index]), float(mean_scenario_auc[best_index])


GLOBAL_FIXED_BLEND_WEIGHT, GLOBAL_FIXED_BLEND_OOF_AUC = select_global_fixed_blend_weight(
    y_train,
    oof_full,
    oof_score_free,
)
print(f'Selected global full-expert weight: {GLOBAL_FIXED_BLEND_WEIGHT:.2f}')
print(f'Mean OOF ROC-AUC: {GLOBAL_FIXED_BLEND_OOF_AUC:.6f}')


## Expert training


In [ ]:
FULL_MODEL_PATH = OUTPUT_DIR / 'full_expert.cbm'
SCORE_FREE_MODEL_PATH = OUTPUT_DIR / 'score_free_expert.cbm'
APPLICATION_MODEL_PATH = OUTPUT_DIR / 'application_only_expert.cbm'

def fit_catboost(path, columns, iterations, seed, label):
    print(f'\nTraining {label}...')
    model = make_catboost(iterations, seed)
    model.fit(X_train[columns], y_train)
    model.save_model(str(path))
    return model

full_expert = fit_catboost(
    FULL_MODEL_PATH, full_feature_columns, BASE_ITERATIONS, SEED, 'full expert'
)
score_free_expert = fit_catboost(
    SCORE_FREE_MODEL_PATH, score_free_columns, BASE_ITERATIONS, SEED + 100, 'score-free expert'
)
application_only_expert = fit_catboost(
    APPLICATION_MODEL_PATH,
    application_only_columns,
    APPLICATION_ONLY_ITERATIONS,
    SEED + 200,
    'application-only expert',
)


## Gate training


In [ ]:
# Build the gate's training rows from OOF predictions. Each OOF prediction was
# produced by a temporary expert that did not train on that applicant.
gate_train_blocks = []
for scenario_number in range(len(SCENARIOS)):
    gate_train_blocks.append(build_gate_features(
        oof_full[:, scenario_number],
        oof_score_free,
        oof_score_availability[:, scenario_number],
        history_context_train,
    ))

gate_train_frame = pd.concat(gate_train_blocks, ignore_index=True)
gate_train_y = np.tile(y_train, len(SCENARIOS)).astype(np.float32)
gate_train_full_p = oof_full.T.reshape(-1).astype(np.float32)
gate_train_score_free_p = np.tile(oof_score_free, len(SCENARIOS)).astype(np.float32)

# Final experts were fitted only on X_train. Their predictions on X_val can
# therefore be used for gate epoch selection without training on validation rows.
validation_score_free_p = score_free_expert.predict_proba(
    X_val[score_free_columns]
)[:, 1].astype(np.float32)
validation_application_p = application_only_expert.predict_proba(
    X_val[application_only_columns]
)[:, 1].astype(np.float32)

validation_full_predictions = {}
validation_score_availability = {}
gate_validation_blocks = []

for scenario_name, cap in SCENARIOS.items():
    validation_scenario_frame, availability = apply_score_scenario(
        X_val,
        maximum_scores=cap,
        applicant_ids=ids_val,
    )
    assert np.array_equal(
        availability,
        validation_scenario_frame[RAW_SCORE_COLUMNS].notna().to_numpy(),
    ), f'Validation mask mismatch in {scenario_name}'

    validation_full_predictions[scenario_name] = full_expert.predict_proba(
        validation_scenario_frame[full_feature_columns]
    )[:, 1].astype(np.float32)
    validation_score_availability[scenario_name] = availability

    gate_validation_blocks.append(build_gate_features(
        validation_full_predictions[scenario_name],
        validation_score_free_p,
        availability,
        history_context_val,
    ))

    del validation_scenario_frame
    clean()

# The vertical order is scenario-major for both train and validation arrays.
gate_validation_frame = pd.concat(gate_validation_blocks, ignore_index=True)
gate_validation_y = np.tile(y_val, len(SCENARIOS)).astype(np.float32)
gate_validation_full_p = np.concatenate([
    validation_full_predictions[scenario]
    for scenario in SCENARIOS
]).astype(np.float32)
gate_validation_score_free_p = np.tile(
    validation_score_free_p,
    len(SCENARIOS),
).astype(np.float32)

# Fit preprocessing on OOF training rows only, then transform validation.
gate_feature_columns = list(gate_train_frame.columns)
assert gate_feature_columns == list(gate_validation_frame.columns)
assert len(gate_feature_columns) == 19

gate_scaler = StandardScaler().fit(
    gate_train_frame[gate_feature_columns]
)
gate_train_X = gate_scaler.transform(
    gate_train_frame[gate_feature_columns]
).astype(np.float32)
gate_validation_X = gate_scaler.transform(
    gate_validation_frame[gate_feature_columns]
).astype(np.float32)

# Confirm that feature, prediction, and target arrays remain aligned.
assert gate_train_X.shape[0] == len(gate_train_y) == len(gate_train_full_p) == len(gate_train_score_free_p)
assert gate_validation_X.shape[0] == len(gate_validation_y) == len(gate_validation_full_p) == len(gate_validation_score_free_p)
assert np.isfinite(gate_train_X).all() and np.isfinite(gate_validation_X).all()
assert np.isfinite(gate_train_full_p).all() and np.isfinite(gate_validation_full_p).all()

# Select the individual comparison expert on validation, never on test.
validation_roc_auc = {}
validation_best_expert = {}
for scenario_name in SCENARIOS:
    validation_roc_auc[scenario_name] = {
        'Full expert': float(roc_auc_score(
            y_val,
            validation_full_predictions[scenario_name],
        )),
        'Score-free expert': float(roc_auc_score(
            y_val,
            validation_score_free_p,
        )),
        'Application-only expert': float(roc_auc_score(
            y_val,
            validation_application_p,
        )),
    }
    eligible_experts = ['Full expert', 'Score-free expert']
    validation_best_expert[scenario_name] = max(
        eligible_experts,
        key=validation_roc_auc[scenario_name].get,
    )

# Save validation predictions and decisions for an auditable handoff.
validation_payload = {
    'ids_val': ids_val,
    'y_val': y_val,
    'score_free_expert': validation_score_free_p,
    'application_only_expert': validation_application_p,
}
for scenario_name in SCENARIOS:
    validation_payload[f'{scenario_name}__full_expert'] = validation_full_predictions[scenario_name]
    validation_payload[f'{scenario_name}__score_availability'] = validation_score_availability[scenario_name]
np.savez_compressed(
    OUTPUT_DIR / 'validation_predictions.npz',
    **validation_payload,
)

validation_results = {
    'roc_auc': validation_roc_auc,
    'best_expert_by_scenario': validation_best_expert,
    'validation_rows': int(len(y_val)),
    'purpose': 'Gate epoch and comparator selection only; no validation rows are used for fitting.',
}
with open(OUTPUT_DIR / 'validation_results.json', 'w') as file:
    json.dump(validation_results, file, indent=2)

del gate_train_blocks, gate_validation_blocks, gate_train_frame, gate_validation_frame
clean()


In [ ]:
class SAGEGate(nn.Module):
    """Return the mixture weight assigned to the full expert."""

    def __init__(self, input_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.ReLU(),
            nn.Dropout(0.05),
            nn.Linear(16, 1),
        )

    def forward(self, features):
        return torch.sigmoid(self.network(features)).squeeze(1)


def mixture_binary_cross_entropy(weights, full_p, score_free_p, labels):
    """Calculate log loss for the probability mixture produced by the gate."""
    mixture = torch.clamp(
        weights * full_p + (1.0 - weights) * score_free_p,
        1e-6,
        1 - 1e-6,
    )
    return nn.functional.binary_cross_entropy(mixture, labels)


def gate_loader(features, full_p, score_free_p, labels, shuffle):
    """Package explicitly supplied arrays without drawing from another partition."""
    dataset = TensorDataset(
        torch.from_numpy(features),
        torch.from_numpy(full_p),
        torch.from_numpy(score_free_p),
        torch.from_numpy(labels),
    )
    return DataLoader(dataset, batch_size=8192, shuffle=shuffle, num_workers=0)


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0

    for features, full_p, score_free_p, labels in loader:
        features, full_p, score_free_p, labels = [
            value.to(DEVICE)
            for value in (features, full_p, score_free_p, labels)
        ]
        optimizer.zero_grad(set_to_none=True)
        loss = mixture_binary_cross_entropy(
            model(features),
            full_p,
            score_free_p,
            labels,
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_gate_loss(model, features, full_p, score_free_p, labels):
    """Evaluate validation loss without changing model parameters."""
    model.eval()
    loader = gate_loader(
        features,
        full_p,
        score_free_p,
        labels,
        shuffle=False,
    )
    total_loss = 0.0

    for feature_batch, full_batch, score_free_batch, label_batch in loader:
        feature_batch, full_batch, score_free_batch, label_batch = [
            value.to(DEVICE)
            for value in (feature_batch, full_batch, score_free_batch, label_batch)
        ]
        loss = mixture_binary_cross_entropy(
            model(feature_batch),
            full_batch,
            score_free_batch,
            label_batch,
        )
        total_loss += loss.item() * len(label_batch)

    return total_loss / len(loader.dataset)


def set_torch_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_gate_epoch():
    """
    Select the epoch count using X_val predictions.

    The training loader contains only OOF X_train rows. Validation rows are
    evaluated after each epoch but are never passed to the optimizer.
    """
    set_torch_seed(SEED + 301)
    model = SAGEGate(gate_train_X.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.003,
        weight_decay=1e-4,
    )
    training_loader = gate_loader(
        gate_train_X,
        gate_train_full_p,
        gate_train_score_free_p,
        gate_train_y,
        shuffle=True,
    )

    best_loss = np.inf
    best_epoch = 1
    stale_epochs = 0
    epoch_history = []

    for epoch in range(1, GATE_MAX_EPOCHS + 1):
        training_loss = train_one_epoch(model, training_loader, optimizer)
        validation_loss = evaluate_gate_loss(
            model,
            gate_validation_X,
            gate_validation_full_p,
            gate_validation_score_free_p,
            gate_validation_y,
        )
        epoch_history.append({
            'epoch': int(epoch),
            'training_loss': float(training_loss),
            'validation_loss': float(validation_loss),
        })
        print(
            f'Epoch {epoch:02d} | training loss {training_loss:.6f} '
            f'| validation loss {validation_loss:.6f}'
        )

        if validation_loss < best_loss - 1e-6:
            best_loss = validation_loss
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= GATE_PATIENCE:
            break

    return int(best_epoch), float(best_loss), epoch_history


def fit_final_gate(epochs):
    """Refit the gate for the selected epoch count using OOF X_train rows only."""
    set_torch_seed(SEED + 302)
    model = SAGEGate(gate_train_X.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.003,
        weight_decay=1e-4,
    )
    training_loader = gate_loader(
        gate_train_X,
        gate_train_full_p,
        gate_train_score_free_p,
        gate_train_y,
        shuffle=True,
    )

    final_training_history = []
    for epoch in range(1, epochs + 1):
        training_loss = train_one_epoch(model, training_loader, optimizer)
        final_training_history.append({
            'epoch': int(epoch),
            'training_loss': float(training_loss),
        })
        print(f'Final gate refit epoch {epoch:02d}/{epochs:02d}')

    return model, final_training_history


def gate_predict(model, scaled_features, full_p, score_free_p, batch_size=32768):
    """Predict the gate weights and resulting SAGE mixture probabilities."""
    model.eval()
    weights = []
    feature_tensor = torch.from_numpy(scaled_features)

    with torch.no_grad():
        for start in range(0, len(feature_tensor), batch_size):
            batch = feature_tensor[start:start + batch_size].to(DEVICE)
            weights.append(model(batch).cpu().numpy())

    weights = np.concatenate(weights)
    mixture = clean_probabilities(
        weights * full_p + (1 - weights) * score_free_p
    )
    return mixture, weights


assert gate_train_X.shape[1] == gate_validation_X.shape[1] == 19

# Validation chooses only the epoch count; it is not added to the final gate fit.
selected_gate_epoch, selected_gate_validation_loss, gate_epoch_history = select_gate_epoch()
sage_gate, final_gate_training_history = fit_final_gate(selected_gate_epoch)

# Save everything required to reload the fitted gate and its preprocessing.
torch.save({
    'state_dict': sage_gate.state_dict(),
    'input_features': gate_train_X.shape[1],
    'gate_feature_columns': gate_feature_columns,
    'selected_gate_epoch': selected_gate_epoch,
    'selected_gate_validation_loss': selected_gate_validation_loss,
    'gate_selection_source': 'held_out_validation_set',
    'alignment_version': 'leakage_safe_train_validation_test',
}, OUTPUT_DIR / 'sage_gate.pt')
joblib.dump(gate_scaler, OUTPUT_DIR / 'gate_scaler.joblib')

essential_settings = {
    'seed': SEED,
    'oof_folds': N_FOLDS,
    'catboost_iterations': BASE_ITERATIONS,
    'gate_max_epochs': GATE_MAX_EPOCHS,
    'gate_patience': GATE_PATIENCE,
    'gate_selection_source': 'held_out_validation_set',
    'selected_gate_epoch': selected_gate_epoch,
    'selected_gate_validation_loss': selected_gate_validation_loss,
    'bootstrap_iterations': BOOTSTRAP_ITERATIONS,
    'global_fixed_blend_weight': GLOBAL_FIXED_BLEND_WEIGHT,
    'global_fixed_blend_oof_auc': GLOBAL_FIXED_BLEND_OOF_AUC,
    'scenarios': SCENARIOS,
    'scenario_ranking_seed': SCENARIO_RANKING_SEED,
    'training_rows': int(len(y_train)),
    'validation_rows': int(len(y_val)),
    'validation_best_expert': validation_best_expert,
    'feature_counts': {
        'full': len(full_feature_columns),
        'score_free': len(score_free_columns),
        'application_only': len(application_only_columns),
    },
    'gate_epoch_history': gate_epoch_history,
    'final_gate_training_history': final_gate_training_history,
    'alignment_version': 'leakage_safe_train_validation_test',
}
with open(OUTPUT_DIR / 'essential_settings.json', 'w') as file:
    json.dump(essential_settings, file, indent=2)

# The trained model, scaler, feature list, and validation-selected comparator remain.
del (
    gate_train_X,
    gate_train_full_p,
    gate_train_score_free_p,
    gate_train_y,
    gate_validation_X,
    gate_validation_full_p,
    gate_validation_score_free_p,
    gate_validation_y,
)
clean()


## Fixed evaluation


In [ ]:
# This is the first post-split cell that processes X_test features. All experts,
# preprocessing, gate parameters, epoch choices, and comparison choices are frozen.
assert len(X_test) == len(y_test) == len(ids_test) == 30_752
history_context_test = build_history_context(X_test)

test_predictions = {}
test_gate_weights = {}
test_score_availability = {}

test_score_free_p = score_free_expert.predict_proba(
    X_test[score_free_columns]
)[:, 1]
test_application_p = application_only_expert.predict_proba(
    X_test[application_only_columns]
)[:, 1]

for scenario, cap in SCENARIOS.items():
    scenario_frame, availability = apply_score_scenario(
        X_test,
        maximum_scores=cap,
        applicant_ids=ids_test,
    )
    assert np.array_equal(
        availability,
        scenario_frame[RAW_SCORE_COLUMNS].notna().to_numpy(),
    ), f'Fixed-evaluation mask mismatch in {scenario}'

    full_p = full_expert.predict_proba(
        scenario_frame[full_feature_columns]
    )[:, 1]
    gate_frame = build_gate_features(
        full_p,
        test_score_free_p,
        availability,
        history_context_test,
    )
    gate_X = gate_scaler.transform(
        gate_frame[gate_feature_columns]
    ).astype(np.float32)
    sage_p, sage_weights = gate_predict(
        sage_gate,
        gate_X,
        full_p,
        test_score_free_p,
    )

    model_predictions = {
        'Full expert': full_p,
        'Score-free expert': test_score_free_p,
        'Application-only expert': test_application_p,
        'Fixed 50/50 mixture': 0.5 * full_p + 0.5 * test_score_free_p,
        'OOF-tuned global fixed blend': (
            GLOBAL_FIXED_BLEND_WEIGHT * full_p
            + (1.0 - GLOBAL_FIXED_BLEND_WEIGHT) * test_score_free_p
        ),
        'SAGE-Credit': sage_p,
    }
    assert all(len(probabilities) == len(y_test) for probabilities in model_predictions.values())
    assert all(
        np.isfinite(probabilities).all()
        and ((probabilities >= 0) & (probabilities <= 1)).all()
        for probabilities in model_predictions.values()
    )

    test_predictions[scenario] = model_predictions
    test_gate_weights[scenario] = sage_weights
    test_score_availability[scenario] = availability

    del scenario_frame, gate_frame, gate_X
    clean()

# Test labels are used only below for final reporting and uncertainty estimates.
test_roc_auc = {
    scenario: {
        model_name: float(roc_auc_score(y_test, probabilities))
        for model_name, probabilities in predictions.items()
    }
    for scenario, predictions in test_predictions.items()
}

test_average_precision = {
    scenario: {
        model_name: float(
            average_precision_score(y_test, probabilities)
        )
        for model_name, probabilities in predictions.items()
    }
    for scenario, predictions in test_predictions.items()
}

test_brier_score = {
    scenario: {
        model_name: float(brier_score_loss(y_test, probabilities))
        for model_name, probabilities in predictions.items()
    }
    for scenario, predictions in test_predictions.items()
}

test_log_loss = {
    scenario: {
        model_name: float(
            log_loss(
                y_test,
                clean_probabilities(probabilities),
                labels=[0, 1],
            )
        )
        for model_name, probabilities in predictions.items()
    }
    for scenario, predictions in test_predictions.items()
}

test_bootstrap = {scenario: {} for scenario in SCENARIOS}
for scenario_number, scenario in enumerate(SCENARIOS):
    for comparison_name in [
        'Full expert',
        'Score-free expert',
        'Application-only expert',
        'Fixed 50/50 mixture',
        'OOF-tuned global fixed blend',
    ]:
        comparison_label = f'SAGE-Credit minus {comparison_name}'
        test_bootstrap[scenario][comparison_label] = paired_auc_bootstrap(
            y_test,
            test_predictions[scenario]['SAGE-Credit'],
            test_predictions[scenario][comparison_name],
            iterations=BOOTSTRAP_ITERATIONS,
            seed=SEED + 1000 + scenario_number,
        )

# Store applicant IDs, targets, predictions, gate weights, and scenario masks.
test_payload = {'ids_test': ids_test, 'y_test': y_test}
for scenario in SCENARIOS:
    for model_name, probabilities in test_predictions[scenario].items():
        safe_name = re.sub(r'[^A-Za-z0-9]+', '_', model_name).strip('_').lower()
        test_payload[f'{scenario}__{safe_name}'] = probabilities
    test_payload[f'{scenario}__gate_weight'] = test_gate_weights[scenario]
    test_payload[f'{scenario}__score_availability'] = test_score_availability[scenario]
np.savez_compressed(OUTPUT_DIR / 'test_predictions.npz', **test_payload)

test_results = {
    'roc_auc': test_roc_auc,
    'average_precision': test_average_precision,
    'brier_score': test_brier_score,
    'log_loss': test_log_loss,
    'paired_auc_bootstrap': test_bootstrap,
    'mean_full_expert_gate_weight': {
        scenario: float(np.mean(test_gate_weights[scenario]))
        for scenario in SCENARIOS
    },
    'test_rows': int(len(y_test)),
    'note': (
        'The test set was used only for final evaluation after all models, '
        'preprocessing, epoch choices, and comparator choices were frozen.'
    ),
}
with open(OUTPUT_DIR / 'test_results.json', 'w') as file:
    json.dump(test_results, file, indent=2)


In [ ]:
test_metric_rows = []

for scenario, predictions in test_predictions.items():
    for model_name in predictions:
        test_metric_rows.append({
            'Scenario': scenario,
            'Model': model_name,
            'ROC-AUC': test_roc_auc[scenario][model_name],
            'Average precision': test_average_precision[scenario][model_name],
            'Brier score': test_brier_score[scenario][model_name],
            'Log loss': test_log_loss[scenario][model_name],
        })

test_metrics_table = pd.DataFrame(test_metric_rows)

display(
    test_metrics_table.style.format({
        'ROC-AUC': '{:.6f}',
        'Brier score': '{:.6f}',
        'Log loss': '{:.6f}',
        'Average precision': '{:.6f}'
    })
)

## Final plots


In [ ]:
# FIGURE 8 - Fixed-Evaluation Performance and History Ablation
# Save the figure with the other experiment artifacts and display it below.

assert "test_roc_auc" in globals(), "Run the fixed-evaluation cell first."

scenario_order = ["natural", "max_2_scores", "max_1_score", "no_scores"]
scenario_labels = [
    "Natural availability",
    "At most 2 scores",
    "At most 1 score",
    "No scores",
]
model_order = [
    "Full expert",
    "Score-free expert",
    "Fixed 50/50 mixture",
    "OOF-tuned global fixed blend",
    "SAGE-Credit",
]

auc_values = np.array([
    [test_roc_auc[scenario][model] for model in model_order]
    for scenario in scenario_order
])
assert auc_values.shape == (4, 5) and np.isfinite(auc_values).all()

application_only_auc = test_roc_auc["no_scores"]["Application-only expert"]
score_free_auc = test_roc_auc["no_scores"]["Score-free expert"]
score_free_minus_application_auc = score_free_auc - application_only_auc

colors = {
    "Full expert": "#0072B2",
    "Score-free expert": "#E69F00",
    "Fixed 50/50 mixture": "#009E73",
    "OOF-tuned global fixed blend": "#56B4E9",
    "SAGE-Credit": "#CC79A7",
}
markers = {
    "Full expert": "o",
    "Score-free expert": "s",
    "Fixed 50/50 mixture": "^",
    "OOF-tuned global fixed blend": "v",
    "SAGE-Credit": "D",
}

fig, (ax_a, ax_b) = plt.subplots(
    1, 2, figsize=(15, 6), dpi=140,
    gridspec_kw={"width_ratios": [1.55, 1]}, facecolor="white"
)
fig.suptitle(
    "Fixed-Evaluation Performance and Historical Feature Fallback",
    fontsize=16, fontweight="semibold"
)

x_positions = np.arange(len(scenario_order))
for model_index, model in enumerate(model_order):
    ax_a.plot(
        x_positions, auc_values[:, model_index],
        color=colors[model], marker=markers[model], markersize=7,
        linewidth=2.2, label=model
    )
ax_a.set_xticks(x_positions)
ax_a.set_xticklabels(scenario_labels)
lower_limit = max(0.50, float(np.floor((auc_values.min() - 0.005) * 100) / 100))
upper_limit = min(1.00, float(np.ceil((auc_values.max() + 0.005) * 100) / 100))
ax_a.set_ylim(lower_limit, upper_limit)
ax_a.set_ylabel("Evaluation ROC-AUC")
ax_a.set_xlabel("Controlled Score Availability Scenario")
ax_a.set_title("A. Performance Across Score Availability Regimes", loc="left")
ax_a.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.65)
ax_a.set_axisbelow(True)
ax_a.legend(frameon=False, loc="lower left", fontsize=9)

ax_b.hlines(0, application_only_auc, score_free_auc, color="#7F7F7F", linewidth=3)
ax_b.scatter(application_only_auc, 0, s=110, color="#D55E00", marker="o", zorder=2)
ax_b.scatter(score_free_auc, 0, s=110, color="#0072B2", marker="D", zorder=2)
ax_b.annotate(
    f"Application only\n{application_only_auc:.6f}",
    (application_only_auc, 0), xytext=(0, -20), textcoords="offset points",
    ha="center", va="top", fontsize=9, color="#A63F00", fontweight="semibold"
)
ax_b.annotate(
    f"Score-free expert\n{score_free_auc:.6f}",
    (score_free_auc, 0), xytext=(0, 15), textcoords="offset points",
    ha="center", va="bottom", fontsize=9, color="#005A8D", fontweight="semibold"
)
ax_b.annotate(
    f"Observed difference: {score_free_minus_application_auc:+.6f}",
    ((application_only_auc + score_free_auc) / 2, 0),
    xytext=(0, 42), textcoords="offset points", ha="center",
    fontsize=10, fontweight="semibold"
)
padding = max(abs(score_free_minus_application_auc) * 0.18, 0.004)
ax_b.set_xlim(min(application_only_auc, score_free_auc) - padding,
              max(application_only_auc, score_free_auc) + padding)
ax_b.set_ylim(-0.35, 0.35)
ax_b.set_yticks([])
ax_b.set_xlabel("Evaluation ROC-AUC")
ax_b.set_title("B. Predictive Value of Engineered Features Without Scores", loc="left", fontsize=11)
ax_b.grid(axis="x", color="#D9D9D9", linewidth=0.8, alpha=0.65)
ax_b.set_axisbelow(True)

fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(OUTPUT_DIR / 'figure7_availability_performance_and_history_ablation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# FIGURE 9 - Fixed-Evaluation Gate Behavior
# Save the figure with the other experiment artifacts and display it below.

assert "test_gate_weights" in globals(), "Run the test evaluation cell first."

scenario_order = ["natural", "max_2_scores", "max_1_score", "no_scores"]
scenario_labels = [
    "Natural availability",
    "At most 2 scores",
    "At most 1 score",
    "No scores",
]
mean_full_expert_weights = np.array([
    float(np.mean(test_gate_weights[scenario])) for scenario in scenario_order
])
assert np.isfinite(mean_full_expert_weights).all()
assert np.all((mean_full_expert_weights >= 0) & (mean_full_expert_weights <= 1))

fig, ax = plt.subplots(figsize=(9.5, 5.8), dpi=140, facecolor="white")
bar_colors = ["#0072B2", "#56B4E9", "#E69F00", "#009E73"]
bars = ax.bar(
    scenario_labels, mean_full_expert_weights,
    color=bar_colors, width=0.68, edgecolor="white", linewidth=0.8
)
ax.axhline(0.5, color="#666666", linestyle="--", linewidth=1.2, alpha=0.8)
for bar, value in zip(bars, mean_full_expert_weights):
    ax.annotate(
        f"{value:.3f}",
        (bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, 7), textcoords="offset points",
        ha="center", va="bottom", fontsize=10, fontweight="semibold"
    )
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean Weight Assigned to Full Expert")
ax.set_xlabel("Controlled Score Availability Scenario")
ax.set_title("SAGE-Credit Gate Behavior on the Fixed Evaluation Set")
ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.65)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure10_gate_behavior_by_availability.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# FIGURE 10 - SAGE-Credit Relative to a Validation-Selected Individual Expert
# The comparator is selected on validation; test data supplies only final estimates.

assert "test_roc_auc" in globals() and "test_bootstrap" in globals(), \
    "Run the fixed-evaluation cell first."
assert set(validation_best_expert) == set(SCENARIOS)

scenario_order = ["natural", "max_2_scores", "max_1_score", "no_scores"]
scenario_short_labels = ["Natural", "At most 2 scores", "At most 1 score", "No scores"]

best_experts = []
observed_deltas = []
ci_lowers = []
ci_uppers = []

for scenario in scenario_order:
    # This choice was frozen using validation ROC-AUC in Cell 17.
    best_expert = validation_best_expert[scenario]
    assert best_expert in {'Full expert', 'Score-free expert'}
    best_experts.append(best_expert)

    sage_auc = test_roc_auc[scenario]['SAGE-Credit']
    comparison_auc = test_roc_auc[scenario][best_expert]
    observed_deltas.append(sage_auc - comparison_auc)

    comparison_label = f'SAGE-Credit minus {best_expert}'
    bootstrap_result = test_bootstrap[scenario][comparison_label]
    ci_lowers.append(float(bootstrap_result['AUC CI lower']))
    ci_uppers.append(float(bootstrap_result['AUC CI upper']))

observed_deltas = np.asarray(observed_deltas)
ci_lowers = np.asarray(ci_lowers)
ci_uppers = np.asarray(ci_uppers)
assert np.all(ci_lowers <= observed_deltas) and np.all(observed_deltas <= ci_uppers)

expert_short = {
    'Full expert': 'Full',
    'Score-free expert': 'Score-free',
}
row_labels = [
    f'{scenario}: {expert_short[expert]}'
    for scenario, expert in zip(scenario_short_labels, best_experts)
]
y_positions = np.arange(len(scenario_order))
left_errors = observed_deltas - ci_lowers
right_errors = ci_uppers - observed_deltas

fig, ax = plt.subplots(figsize=(11, 6.2), dpi=140, facecolor='white')
ax.errorbar(
    observed_deltas,
    y_positions,
    xerr=np.vstack([left_errors, right_errors]),
    fmt='o',
    markersize=8,
    color='#0072B2',
    ecolor='#0072B2',
    elinewidth=2.2,
    capsize=5,
    capthick=1.6,
)
ax.axvline(0, color='#555555', linestyle='--', linewidth=1.2)
for y_position, delta, upper in zip(y_positions, observed_deltas, ci_uppers):
    ax.annotate(
        f'{delta:+.6f}',
        (upper, y_position),
        xytext=(7, 0),
        textcoords='offset points',
        ha='left',
        va='center',
        fontsize=9.5,
        fontweight='semibold',
    )

ax.set_yticks(y_positions)
ax.set_yticklabels(row_labels)
ax.invert_yaxis()
ax.margins(y=0.18)
span = ci_uppers.max() - ci_lowers.min()
padding = max(span * 0.14, 0.0006)
ax.set_xlim(ci_lowers.min() - padding, ci_uppers.max() + 2.8 * padding)
ax.set_xlabel('Test ROC-AUC Difference: SAGE-Credit Minus Validation-Selected Expert')
ax.set_title('SAGE-Credit Relative to the Validation-Selected Expert on the Untouched Test Set')
ax.grid(axis='x', color='#D9D9D9', linewidth=0.8, alpha=0.65)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / 'figure8_best_expert_bootstrap_comparison.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()


In [ ]:
# Visualize which feature families are available to each fitted expert.
expert_names = ['Application-only expert', 'Score-free expert', 'Full expert']
application_counts = np.array([len(application_only_columns)] * 3)
historical_counts = np.array([0, len(history_columns), len(history_columns)])
score_counts = np.array([0, 0, len(ALL_SCORE_COLUMNS)])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(expert_names, application_counts, label='Application features')
ax.barh(
    expert_names,
    historical_counts,
    left=application_counts,
    label='Engineered features',
)
ax.barh(
    expert_names,
    score_counts,
    left=application_counts + historical_counts,
    label='External-score features',
)
ax.set_xlabel('Number of model features')
ax.set_title('Features seen by each expert')
ax.legend()
fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / 'figure5_expert_feature_composition.png',
    dpi=300,
    bbox_inches='tight',
)
plt.show()
